<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Meet the inbox

**[Meet the inbox](https://learning.nextia-ai.com/courses/text-routing/m01/meet-the-inbox/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 1, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** open the Larkfield inbox, read its labels and its cost table, and run the first comparison of the course: a person for everything, keyword rules, and a first model.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The routes and the cost of each mistake, from [Design the labels](https://learning.nextia-ai.com/courses/text-routing/m01/design-the-labels/) and [The cost of each mistake](https://learning.nextia-ai.com/courses/text-routing/m01/the-cost-of-each-mistake/). |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m01/meet-the-inbox/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M01-L04-meet-the-inbox/meet-the-inbox-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "fce0d2e602002a2ea2c6fc26c1d2ab9c3d1a30906f40fa4a426180c3639e99af",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "8ab83de04998b9017b1aabce00be04084e989af75aff5a9abdd760f4d3cb24b2",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "54c2e724cb3c855a0fa6302d4ac0ab2df2ffaf788eec4ab198c6feb8d3508928",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. What is in the inbox

Each row of `messages.csv` is one message that a customer sent to Larkfield's support inbox. The columns that this course uses most:

- `text`: what the customer wrote, in English or French;
- `channel`: `email`, `chat` or `social`; `language`: `en` or `fr`;
- `route`: the team that must act first, or `needs_person` (the label);
- `topic` and `priority`: the second and third questions about the same message;
- `split`: `train`, `validation` or `test`.

> **Predict.** Which route has the most messages? Which has the fewest? Write your guess, then run the cell.

In [ ]:
print(len(messages), "messages from", messages["created_at"].min()[:10], "to", messages["created_at"].max()[:10])
print(messages["route"].value_counts().to_string())

> **Check.** You should see 10000 messages, `delivery` first with 2318 messages and `needs_person` last with 491.

The routes are not the same size. A model that learns from these messages sees `delivery` many times and `needs_person` only a few hundred times. Module 7 comes back to this.

## 2. One message from each route

The next cell shows one message per route from the training messages, with its language and channel.

In [ ]:
train.groupby("route").head(1)[["route", "language", "channel", "text"]]

Read the messages. Which words tell you the route? Some messages say the route in clear words ("refund", "charged twice"). Others do not: the customer tells a story, and you must find the request.

## 3. Two questions, one message

A message has a route, a topic and a priority. The next cell counts the topics inside each route, and the priority of the messages.

In [ ]:
print(train.groupby("route")["topic"].nunique().to_string())
print(train["priority"].value_counts().to_string())
print("Account takeover messages in train:", (train["topic"] == "account_takeover").sum())

> **Check.** You should see 27 topics in all, and 44 account takeover messages in the training part: a rare class.

## 4. The cost table

Grace and Priya agreed on what each kind of mistake costs. These are **assumptions**, not measurements: `data/costs.json` gives the reason for each one.

In [ ]:
costs = rt.load_costs()
for name, dollars in costs.items():
    print(f"{name:30} {dollars:8.2f} dollars")
print(rt.message_cost("returns_refunds", "product", "normal", "refund_status"))
print(rt.message_cost("account", "billing", "urgent", "account_takeover"))

> **Check.** The last two lines are 4.0 (a refund request in the product team's queue) and 200.0 (an account takeover in the billing queue).

## 5. The first comparison

Now the course's first comparison, on the 2,000 test messages:

1. **Everything to a person.** Larkfield today: a person reads each message and sends it on (1.50 dollars a message).
2. **Rules or a person.** Keyword rules decide when one matches; a person reads the rest.
3. **TF-IDF + logistic regression.** A first model, trained on the 6,000 training messages. Module 3 explains it.

> **Predict.** Which one has the lowest total cost per 1,000 messages? Is it also the most accurate?

In [ ]:
import re, time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, FeatureUnion

# The keyword rules of the course project (data/rules.json): the first rule that matches decides.
RULES = [(r["route"], re.compile(r["pattern"], re.I)) for r in json.load(open("data/rules.json"))["rules"]]

def by_rules(text):
    for route, pattern in RULES:
        if pattern.search(text):
            return route
    return "review"

model = make_pipeline(
    FeatureUnion([("words", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")),
                  ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True))]),
    LogisticRegression(C=10, max_iter=3000))
model.fit(train["text"], train["route"])

rows = [rt.score("everything to a person", test, ["review"] * len(test)),
        rt.score("rules or a person", test, [by_rules(t) for t in test["text"]]),
        rt.score("TF-IDF + logistic regression", test, model.predict(test["text"]))]
rt.show(rows)

> **Check.** You should see three rows. TF-IDF + logistic regression has accuracy 0.962 and a total of 296.0 dollars per 1,000 messages; everything to a person costs 1500.0.

The project's `route compare --quick` prints the same three rows: it uses the same rules (`config/rules.json` in the project) and the same model. Read the columns: `takeover` is the share of account takeovers that reach the account team, and `EN` and `FR` are the accuracy in English and in French.

## 6. Your turn: the French messages

> **Your turn.** Count the test messages in French that the model routes to the wrong team. Put the number in `wrong_fr`. Use `model.predict(...)` on the French test messages and compare with their `route`.

In [ ]:
wrong_fr = ...  # the number of French test messages that the model gets wrong

In [ ]:
expect('wrong_fr', wrong_fr, 28)

## 7. Failure case: accuracy without the cost

Tomás says: "The model is right on 96% of the messages. Ship it." The next cell shows where the model's mistakes cost the most: the wrong routes, with their cost.

In [ ]:
pred = model.predict(test["text"])
wrong = test[pred != test["route"]].assign(predicted=pred[pred != test["route"]])
wrong["cost"] = [rt.message_cost(t, p, pr, tp) for t, p, pr, tp in
                 zip(wrong["route"], wrong["predicted"], wrong["priority"], wrong["topic"])]
print(wrong.groupby("route")["cost"].agg(["count", "sum"]).sort_values("sum", ascending=False).to_string())

A few expensive mistakes (a safety message in a team's queue, a missed account takeover) can cost more than many cheap ones. That is why the course compares methods by the **cost per 1,000 messages**, not by accuracy alone.

## Reset

If you change files in `larkfield-inbox` and want to start again, delete the folder and run the setup cells again. In the project on your computer, `route reset` does the same.

## Next

You opened the inbox and ran the first comparison. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m01/meet-the-inbox/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Labelling guidelines](https://learning.nextia-ai.com/courses/text-routing/m02/labelling-guidelines/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m01/meet-the-inbox/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.